# Lab Notes: Multi-Agent Orchestration & Workflow Topologies
## Deconstructing Specialist Collaboration, Deterministic Pipelines, and Directed Workflow Graphs in Google ADK

In basic agent implementations, a single agent handles all reasoning, tool execution, and response synthesis. While effective for localized tasks, this **"monolithic agent"** pattern breaks down rapidly under real-world complexity:
* **Context Dilution:** When prompts include instructions for multiple distinct domains, the model's instruction-following fidelity degrades.
* **Conflicting System Instructions:** A persona tailored for creative ideation often clashes with the strict token efficiency required for technical summarization or code synthesis.
* **Fragile Error Isolation:** If a single sub-task fails, the entire turn fails, with zero observability into which cognitive step broke down.

### The Paradigm: Modular Multi-Agent Systems
Instead of a single monolithic model, we design **multi-agent topologies**: ecosystems of discrete, single-purpose agents that collaborate across shared state.

### Orchestration Paradigms Explored in this Lab:
1. **Dynamic LLM-Driven Orchestration (Supervisor-Subagent):** A central coordinator decides dynamically which specialist agents to invoke as callable tools.
2. **Deterministic Pipelines (Sequential Agent):** A linear assembly line where output from stage $N$ deterministically flows into stage $N+1$.
3. **Concurrent Execution & Fan-In (Parallel Agent + Aggregator):** Executing independent subtasks simultaneously to eliminate latency bottlenecks.
4. **Modern Directed Graphs (`google.adk.workflow.Workflow`):** Moving beyond legacy wrapper agents to explicit stateful graphs with programmatic routing (`JoinNode`, conditional `@node` routers).
5. **Evaluator-Optimizer Cycles (Loop Agent vs. Cyclic Graph):** Iterative generation and critique loops to guarantee quality standards.

In [7]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key
google_api_key = os.getenv("GOOGLE_API_KEY")

# Verify it was loaded successfully (masks key for security)
if google_api_key:
    print(f"Loaded successfully: {google_api_key[:6]}...{google_api_key[-4:]}")
else:
    print("GOOGLE_API_KEY not found in .env")


Loaded successfully: AIzaSy...TC0w


In [3]:
# Configure Retry Options
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

### 1. Pattern 1: Dynamic LLM Coordination (Supervisor-Subagent Pattern)

#### Architectural Concept
In this topology, a parent agent (`ResearchCoordinator`) acts as an intelligent supervisor. Rather than hardcoding the sequence of steps in Python code, the supervisor is given access to specialized sub-agents wrapped as tools via `AgentTool(...)`.

![Multi-agent Team](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png)

#### System Design & State Plumbing:
* **`ResearchAgent`:** Uses `google_search` to gather verified domain knowledge. Stores its output in the shared session context under `output_key="research_findings"`.
* **`SummarizerAgent`:** Ingests the state variable via template interpolation (`{research_findings}`) and produces a structured summary under `output_key="final_summary"`.
* **`ResearchCoordinator`:** Equipped with `tools=[AgentTool(research_agent), AgentTool(summarizer_agent)]`. It reasons over user requests and decides when and in what order to invoke each sub-agent.

> **Trade-Off Analysis:**
> * **Advantage:** Maximum flexibility. The coordinator can call specialists dynamically, skip unnecessary steps, or loop based on conversational context.
> * **Failure Mode:** Stochastic execution order. Relying solely on prompt instructions to enforce multi-step ordering can lead to skipped steps or non-deterministic executions.

In [4]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [5]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


In [8]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [9]:
runner = InMemoryRunner(agent=root_agent)
runner = await runner.run_debug(
    "What is latest advancements in AI, and what is the best way to learn about them?"
)

c:\Users\devas\Documents\Development\AgentDevCourse\.venv\Lib\site-packages\google\adk\models\llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


ResearchCoordinator > Here is a comprehensive overview of the latest advancements in Artificial Intelligence and the best ways to get started or level up your knowledge:

---

### **Latest Advancements in AI**
The AI landscape has evolved far beyond basic text chats and simple automations, shifting heavily toward reasoning, autonomy, and deep integration:
1. **Reasoning Models (System 2 Thinking):** Newer AI architectures break complex queries down into multi-step logical paths before answering, mimicking human problem-solving.
2. **Native Multimodality:** State-of-the-art models process text, high-resolution video, audio, and code simultaneously in real-time.
3. **Massive Context Windows:** Models can now process millions of tokens in a single prompt—allowing you to drop entire codebases or multi-volume books directly into a chat.
4. **Edge AI & Small Language Models (SLMs):** Highly optimized lightweight models now run directly on smartphones and local hardware for instant, privacy-f

### 2. Pattern 2: Deterministic Linear Pipelines (`SequentialAgent`)

#### Architectural Concept
When task steps have strict causal dependencies ($A \longrightarrow B \longrightarrow C$), delegating execution order to LLM reasoning introduces unnecessary risk. 

The `SequentialAgent` enforces a deterministic **assembly line** pipeline where sub-agents execute in exact sequence. State is passed downstream through session state variables defined by `output_key`.

![Sequential Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png)

#### Pipeline Implementation: Three-Stage Content Engine
1. **`OutLineAgent`:** Conducts initial research via `google_search` and structures a 4-part blog outline $\longrightarrow$ writes to `output_key="blog_outline"`.
2. **`WriterAgent`:** Ingests `{blog_outline}`, drafts a focused 400-500 word narrative $\longrightarrow$ writes to `output_key="blog_draft"`.
3. **`EditorAgent`:** Ingests `{blog_draft}`, polishes grammar, enhances tonal flow, and refines clarity $\longrightarrow$ writes to `output_key="final_blog"`.

```
[User Prompt] ───> ( OutLineAgent ) ──[blog_outline]──> ( WriterAgent ) ──[blog_draft]──> ( EditorAgent ) ──[final_blog]──> [Final Output]
```

> **Engineering Takeaway:**
> Linear pipelines guarantee execution order, making them ideal for multi-stage content drafting, ETL data transformation, and structured document processing.

In [11]:
outline_agent = Agent(
    name = "OutLineAgent" ,
    model = Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
        instruction = """ You are an agent to create outline for a blog post. Your job is to:
        1. A catchy headline
        2. An introductory hook
        3. A main section with 3-5 key ideas with 2-3 bullet points for each that can be identified using google search.
        4. A concluding thought that provides an aha moment.
        """,
        tools= [google_search],
        output_key="blog_outline" # The result of this agent will be stored in the session state with this key.
)
print("outline_agent is created")

outline_agent is created


In [12]:
writer_agent = Agent(
    name = "WriterAgent" ,
    model=Gemini(
        model="gemini-3.5-flash-lite" ,
        retry_options=retry_config 
    ),
    instruction= """ 
    Your job is to write the blog post strictly on the {blog_outline}.
    Write a brief 400-500 words blog post with an engaging and informative tone. 
    Make sure to not use any em-dash in the blog post.
    """ ,
    output_key="blog_draft",
)
print("writer_agent is created")

writer_agent is created


In [13]:
editor_agent = Agent(
    name = "EditorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    instruction= """ 
    Edit this draft: {blog_draft}.
    Polish the draft by fixing grammatical errors, sentence structure, improving the flow and enhancing clarity.
    """ ,
    output_key='final_blog',
)
print("editor_agent is created")

editor_agent is created


In [15]:
root_agent = SequentialAgent(
    name = "BlogPipeline" ,
    sub_agents=[outline_agent,writer_agent,editor_agent],
)
print("Sequential Agent Created")

Sequential Agent Created


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\3931043205.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [16]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post on how Hurufism is still practised in contemporary pop culture"
)

OutLineAgent > Here is a comprehensive outline for a blog post exploring how Hurufism manifests in contemporary pop culture:

# Decoding the Cosmos: How 14th-Century Hurufism Secretly Shapes Today’s Pop Culture

### 1. Introductory Hook
Imagine looking at a sprawling street mural, a piece of avant-garde graphic design, or a conceptual music video, feeling a deep sense of modern aesthetic wonder—without realizing you are staring at a coded mystical philosophy born in the 14th century. While medieval Islamic mysticism is rarely talked about over morning coffee, the ancient philosophy of **Hurufism**—the belief that letters and numbers hold the esoteric key to the universe—is quietly alive and well. Far from being confined to dusty historical archives, Hurufism’s obsession with the power of letters continues to pulse beneath the surface of contemporary art, design, and pop culture.

---

### 2. Main Section: Key Ideas and Supporting Points

*   **Idea 1: The *Hurufiyya* Movement and Conte

### 3. Pattern 3: Concurrent Fan-Out & Fan-In Aggregation (`ParallelAgent`)

#### Architectural Concept
Sequential execution creates an unnecessary latency bottleneck when sub-tasks are mutually independent. 

The `ParallelAgent` executes multiple specialist agents concurrently. Once all parallel executions complete, downstream nodes synthesize the combined outputs into a cohesive deliverable.

![Parallel Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png)

#### Multi-Sector Intelligence Briefing Architecture:
* **Concurrent Fan-Out (`ParallelAgent`):**
  * `TechResearcher` $\longrightarrow$ extracts AI/ML trends into `output_key="tech_output"`.
  * `HealthResearcher` $\longrightarrow$ extracts medical breakthroughs into `output_key="health_output"`.
  * `FinanceResearcher` $\longrightarrow$ extracts fintech innovations into `output_key="finance_output"`.
* **Sequential Synthesis (`SequentialAgent` wrapper):**
  * `AggregatorAgent` $\longrightarrow$ ingests `{tech_output}`, `{health_output}`, and `{finance_output}` to identify cross-sector intersections and produce an executive briefing.

```
                  ┌──> [ TechResearcher   ] ──(tech_output)────┐
                  │                                            │
[User Request] ───┼──> [ HealthResearcher ] ──(health_output)──┼──> [ AggregatorAgent ] ──> [Executive Briefing]
                  │                                            │
                  └──> [ FinanceResearcher] ──(finance_output)──┘
```

In [30]:
tech_researcher = Agent(
    name = "TechResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest AI/ML trends. Include 3 key developments, 
    the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """ ,
    output_key='tech_output',
    tools=[google_search],
)

print("tech_researcher Agent is created!")

tech_researcher Agent is created!


In [31]:
health_researcher = Agent(
    name = "HealthResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest Health trends and medical advances. Include 3 significant advances, 
    their practical application, and the estimated timeline. Keep the report very concise (100 words).
    """ ,
    output_key='health_output',
    tools=[google_search],
)

print("health_researcher Agent is created!")

health_researcher Agent is created!


In [32]:
finance_researcher = Agent(
    name = "FinanceResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research current fintech trends. Include 3 key trends,
    their market implications, and the future outlook. Keep the report concise (100 words).
    """ ,
    output_key='finance_output',
    tools=[google_search],
)

print("finance_researcher Agent is created!")

finance_researcher Agent is created!


In [33]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.

aggregator_agent = Agent(
    name= "AggregatorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction= """ 
    Combine the three research findings into a single executive summary:
    ** Technology Trends: ** 
    {tech_output}

    ** Heath Breakthroughs: **
    {health_output}

    ** Finance Innovations: **
    {finance_output}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. 
    The final summary should be around 200 words.
    """ ,
    output_key="executive_summary"
)

print("aggregator_agent is Created!")

aggregator_agent is Created!


In [34]:
# The ParallelAgent runs all its sub-agents simultaneously.
# This design ensures that the research agents run first in parallel, then once all of their research is complete, 
# the aggregator agent brings together all of the research findings into a single report

parallel_research_team = ParallelAgent(
    name = "ParallelResearchTeam",
    sub_agents = [tech_researcher,finance_researcher,health_researcher]

)

root_agent = SequentialAgent(
    name = "ResearchSystem" ,
    sub_agents=[parallel_research_team,aggregator_agent] ,
)

print("Parallel & Sequential Agents created")

Parallel & Sequential Agents created


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\2245213889.py:5: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_team = ParallelAgent(
C:\Users\devas\AppData\Local\Temp\ipykernel_31228\2245213889.py:11: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [35]:
runner = InMemoryRunner(root_agent)
response = await runner.run_debug(
    "Run the Daily Executive briefing on Tech, Health & Finance"
) 

TechResearcher > **Daily Executive Briefing: AI/ML Trends**

**Key Developments:**
1. **Agentic AI in Production:** Systems are evolving from passive text generators into autonomous agents capable of planning, executing multi-step workflows, and utilizing software tools independently.
2. **AI for Scientific Discovery:** Machine learning is heavily accelerating research workflows, driving complex engineering simulations and novel chemical/drug molecule discovery.
3. **Massive Infrastructure Scaling:** Enterprise architectures are prioritizing heavy investments in dedicated power grids, specialized chips, and cybersecurity to support continuous AI operations.

**Main Companies Involved:**
* OpenAI, Microsoft, Google, and enterprise cloud providers scaling autonomous software capabilities.

**Potential Impact:**
Drastically increases operational efficiency and automates complex backend workflows across finance, health, and tech, while demanding major capital updates for energy and digital

### 4. Modernizing Orchestration: Migrating from Legacy Agents to ADK `Workflow`

#### The Architectural Transition
In modern versions of Google ADK, high-level wrapper agents (`SequentialAgent`, `ParallelAgent`) are deprecated in favor of **`google.adk.workflow.Workflow`**. 

`Workflow` introduces explicit **Directed Graph Topologies**, providing complete control over node lifecycles, branching logic, synchronization barriers, and error recovery.

#### Syntax & Graph Primitives:
* **Linear / Sequential Pipeline:** `edges=[ (START, agent_a, agent_b, agent_c) ]`
* **Parallel Fan-Out:** `edges=[ (START, (agent_1, agent_2, agent_3)) ]`
* **Synchronized Fan-In:** `edges=[ (START, (agent_1, agent_2, agent_3), join_node, aggregator) ]`

---

### ⚠️ Critical Engineering Insights & Debugging Notes:

#### 1. The `mode="single_turn"` Constraint
* **The Error:**
  `ValueError: The agent 'WriterAgent' has been added to the workflow with mode='chat' following node 'OutLineAgent'.`
* **Root Cause:** By default, `Agent` instances operate in conversational `mode="chat"`, expecting an active human-in-the-loop chat session. In graph workflows, nodes act as discrete processing transformations consuming upstream state.
* **Solution:** Explicitly set `mode="single_turn"` on all downstream workflow agents.

#### 2. Parallel Race Conditions & The `JoinNode` Barrier
* **The Error:**
  `Context variable not found: 'health_output' in agent 'AggregatorAgent'.`
* **Root Cause:** In standard workflow graphs, nodes fire as soon as their *first* predecessor resolves. In a parallel tuple `(tech, health, finance)`, whichever researcher completes first immediately triggers the downstream aggregator before the other agents have written their state keys.
* **Solution:** Insert an explicit `JoinNode(name="join")` as a synchronization barrier:
  ```python
  join = JoinNode(name="join")
  edges = [(START, (tech_researcher, health_researcher, finance_researcher), join, aggregator_agent)]
  ```
  `JoinNode` blocks execution until **all** branches in the tuple have successfully written to state.

In [ ]:
#### Replacing Sequential Agent with Workflow()

from google.adk.workflow import Workflow, START

outline_agent = Agent(
    name = "OutLineAgent" ,
    model = Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
        instruction = """ You are an agent to create outline for a blog post. Your job is to:
        1. A catchy headline
        2. An introductory hook
        3. A main section with 3-5 key ideas with 2-3 bullet points for each that can be identified using google search.
        4. A concluding thought that provides an aha moment.
        """,
        tools= [google_search],
        output_key="blog_outline" # The result of this agent will be stored in the session state with this key.
)
print("outline_agent is created")

writer_agent = Agent(
    name = "WriterAgent" ,
    model=Gemini(
        model="gemini-3.5-flash-lite" ,
        retry_options=retry_config 
    ),
    instruction= """ 
    Your job is to write the blog post strictly on the {blog_outline}.
    Write a brief 400-500 words blog post with an engaging and informative tone. 
    Make sure to not use any em-dash in the blog post.
    """ ,
    output_key="blog_draft",
    mode="single_turn" , # Newly added, not existing in the earlier version
)
print("writer_agent is created")

editor_agent = Agent(
    name = "EditorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    instruction= """ 
    Edit this draft: {blog_draft}.
    Polish the draft by fixing grammatical errors, sentence structure, improving the flow and enhancing clarity.
    """ ,
    output_key='final_blog',
    mode="single_turn" , # Newly added, not existing in the earlier version
)
print("editor_agent is created")

#Replace Sequential Workflow in the Blog Pipeline:
root_agent = Workflow(
    name = "BlogPipeline" ,
    edges=[
        (START,outline_agent,writer_agent,editor_agent)
    ]
)

print("Sequential Workflow Created!")

runner = InMemoryRunner(root_agent)
response = await runner.run_debug(
    "Create a blog post on whether India uses Asymmetrical warfare to counter China in Defence capabilities"
)

outline_agent is created
writer_agent is created
editor_agent is created
Sequential Workflow Created!
OutLineAgent > ### 1. Catchy Headline
**David vs. Dragon: Is India Using Asymmetric Warfare to Outsmart China’s Defense Machine?**

### 2. Introductory Hook
When looking at the sheer metrics of military might, matching China weapon-for-weapon, ship-for-ship, or yuan-for-dollar is a mathematical losing game for India. With Beijing possessing a vastly superior defense budget, advanced industrial capacity, and cutting-edge informationized warfare infrastructure, a symmetrical arms race is practically out of reach. Yet, military strategy isn't just about counting numbers—it’s about turning geography, ingenuity, and unconventional tactics into leverage. As geopolitical tensions persist along the Line of Actual Control (LAC), a critical question arises: Is India quietly leveraging asymmetric warfare to level the playing field against its northern neighbor?

---

### 3. Main Section: Key Idea

In [46]:
#### Replacing Parallel Agent with Workflow()

from google.adk.workflow import Workflow, START , JoinNode

tech_researcher = Agent(
    name = "TechResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest AI/ML trends. Include 3 key developments, 
    the main companies involved, and the potential impact. Keep the report very concise (100 words).
    """ ,
    output_key='tech_output',
    tools=[google_search],
    mode='single_turn',
)

print("tech_researcher Agent is created!")

health_researcher = Agent(
    name = "HealthResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research the latest Health trends and medical advances. Include 3 significant advances, 
    their practical application, and the estimated timeline. Keep the report very concise (100 words).
    """ ,
    output_key='health_output',
    tools=[google_search],
    mode='single_turn',
)

print("health_researcher Agent is created!")

finance_researcher = Agent(
    name = "FinanceResearcher" ,
    model= Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
    Research current fintech trends. Include 3 key trends,
    their market implications, and the future outlook. Keep the report concise (100 words).
    """ ,
    output_key='finance_output',
    tools=[google_search],
    mode='single_turn',
)

print("finance_researcher Agent is created!")

# The AggregatorAgent runs *after* the parallel step to synthesize the results.

aggregator_agent = Agent(
    name= "AggregatorAgent" ,
    model = Gemini(
        model = "gemini-3.5-flash-lite" ,
        retry_options=retry_config
    ) ,
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction= """ 
    Combine the three research findings into a single executive summary:
    ** Technology Trends: ** 
    {tech_output}

    ** Heath Breakthroughs: **
    {health_output}

    ** Finance Innovations: **
    {finance_output}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all three reports. 
    The final summary should be around 200 words.
    """ ,
    output_key="executive_summary",
    mode="single_turn",
)

print("aggregator_agent is Created!")

# 1. Create a JoinNode that waits for all parallel agents to finish
join = JoinNode(name="join")

# 2. Place 'join' between the parallel tuple and the aggregator
root_agent = Workflow(
    name = "ResearchSystem" ,
    edges=[
        (START,(tech_researcher,health_researcher,finance_researcher),join, aggregator_agent) 
    ],
)


print("Parallel Research Workflow created")

runner = InMemoryRunner(root_agent)
response = await runner.run_debug("Run a 2025 briefing on tech, health and finance!")

tech_researcher Agent is created!
health_researcher Agent is created!
finance_researcher Agent is created!
aggregator_agent is Created!
Parallel Research Workflow created
TechResearcher > **Tech, Health & Finance Briefing**

*   **Key Developments:** 
    1. **Agentic AI Expansion:** Autonomous AI agents are scaling across tech and finance to automate complex workflows and real-time decision-making.
    2. **Clinical GenAI Integration:** Healthcare systems are heavily deploying generative AI for documentation automation, claims processing, and drug discovery.
    3. **Automated Compliance & Risk:** Financial institutions are leveraging real-time AI modeling to navigate stringent regulations and secure transactions.
*   **Main Companies:** OpenAI, NVIDIA, J.P. Morgan, Microsoft, and specialized health-tech innovators.
*   **Potential Impact:** Drastically reduced administrative overheads, enhanced patient and customer experiences, and optimized risk management.
HealthResearcher > **Tech

### 5. Pattern 4: Evaluator-Optimizer Feedback Loops (`LoopAgent`)

#### Architectural Concept
One-shot generation often fails to meet strict quality criteria. Complex reasoning, creative drafting, and technical code synthesis benefit significantly from **iterative critique and refinement cycles**.

A `LoopAgent` executes sub-agents repeatedly until an exit condition is triggered or `max_iterations` is reached.

![Loop Agent](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png)

#### Legacy Refinement Pattern:
1. **`InitialWriterAgent`:** Generates the initial story draft into `output_key="current_story"`.
2. **`StoryRefinementLoop` (`LoopAgent`):**
   * **`CriticAgent`:** Reviews `{current_story}`. If acceptable, outputs `"APPROVED"`; otherwise outputs actionable revisions into `output_key="critique"`.
   * **`RefinerAgent`:** Analyzes `{critique}`. If revisions are required, rewrites `{current_story}`; if `"APPROVED"`, calls a tool function: `exit_loop()`.

> **Limitation of `LoopAgent`:**
> Terminating the loop requires the LLM itself to invoke an external function (`exit_loop`). If the model fails to call the tool or hallucinate approval syntax, the loop spins until forced termination by `max_iterations`.

In [61]:
# This Agent runs once at the beginning to create the Story Draft:
initial_writer_agent = Agent(
    name = 'InitialWriterAgent' ,
    model= Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        Based on the user prompt write the first draft of a short story around 100-150 words. 
        Output only story text without an introduction or explanation.  """ ,
    output_key= "current_story"
)
print("Initial Writer Agent Created!")

#This Agent's job is to provide feedback or approval signal. It does not use any tools:
critic_agent = Agent(
    name = 'CriticAgent',
    model=Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        You are a constructive story critic. Review the story provided below.
        Story: {current_story}
    
        Evaluate the story's plot, characters, and pacing.
        - If the story is well-written and complete, and has a moral then you MUST respond with the exact phrase: "APPROVED"
        - Otherwise, provide 2-3 specific, actionable suggestions for improvement.
        """ ,
    output_key="critique"
)

print("critic_agent is created!")


Initial Writer Agent Created!
critic_agent is created!


In [62]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", identify the moral of the story and add it to the story then you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ exit_loop function created.
✅ refiner_agent created.


In [63]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="StoryPipeline",
    sub_agents=[initial_writer_agent, story_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


C:\Users\devas\AppData\Local\Temp\ipykernel_31228\1839283766.py:2: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  story_refinement_loop = LoopAgent(
C:\Users\devas\AppData\Local\Temp\ipykernel_31228\1839283766.py:9: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


In [64]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a short story about a train driver who drove trains for over 20 years"
)

InitialWriterAgent > For twenty-three years, Arthur’s world was framed by a windshield and a ribbon of steel. He knew every curve of the Pennine route, every sleepy whistle-stop, and how the morning mist clung to the tracks near the old viaduct. 

Today was his final run. As he pulled the lever of the diesel engine, the familiar hum vibrated through his bones. Passengers bustled on the platform, unaware they were riding alongside history. Arthur looked out at the fading autumn light, feeling a profound mixture of sorrow and peace. 

He didn't need the speedometer; he felt the train's rhythm in his pulse. When they reached the final terminus, he cut the engine. The sudden silence was deafening. Arthur stood up, adjusted his cap one last time, and stepped out into a future without rails.
CriticAgent > APPROVED
RefinerAgent > For twenty-three years, Arthur’s world was framed by a windshield and a ribbon of steel. He knew every curve of the Pennine route, every sleepy whistle-stop, and how

### 6. Advanced Architecture: Programmatic Loop Routing with `Workflow`

#### Eliminating Artificial Tool Calls
By leveraging `google.adk.workflow.Workflow` alongside the `@node` decorator, we eliminate the need for the agent to call an artificial `exit_loop` tool. Instead, loop termination is handled deterministically via Python routing logic.

#### Key Architectural Refinements:
1. **Deterministic Guard Node (`check_critique`):**
   * Inspects `ctx.session.state.get("critique")` directly.
   * Tracks cycle iteration counters in session state (`ctx.session.state["loop_count"]`).
   * Emits explicit routing signals: `Event(route="exit")` when approved or count $\ge 2$, otherwise `Event(route="refine")`.
2. **Lean `RefinerAgent`:**
   * Stripped of tool dependencies (`tools=[exit_loop]` removed).
   * Operates purely as a revision specialist with `mode="single_turn"`, invoked only when the router selects `"refine"`.
3. **Explicit Conditional Edges:**
   ```python
   edges = [
       (START, initial_writer_agent, critic_agent, check_critique),
       (check_critique, {"refine": refiner_agent, "exit": finish}),
       (refiner_agent, critic_agent),
   ]
   ```

In [65]:
# This Agent runs once at the beginning to create the Story Draft:
initial_writer_agent = Agent(
    name = 'InitialWriterAgent' ,
    model= Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        Based on the user prompt write the first draft of a short story around 100-150 words. 
        Output only story text without an introduction or explanation.  """ ,
    output_key= "current_story",   
)
print("Initial Writer Agent Created!")

#This Agent's job is to provide feedback or approval signal. It does not use any tools:
critic_agent = Agent(
    name = 'CriticAgent',
    model=Gemini(
        model = "gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction= """ 
        You are a constructive story critic. Review the story provided below.
        Story: {current_story}
    
        Evaluate the story's plot, characters, and pacing.
        - If the story is well-written and complete, and has a moral then you MUST respond with the exact phrase: "APPROVED"
        - Otherwise, provide 2-3 specific, actionable suggestions for improvement.
        """ ,
    output_key="critique",
    mode="single_turn"
)

print("critic_agent is created!")

from google.adk.workflow import Workflow, START, node
from google.adk.events import Event

refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-3.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and a critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Rewrite and improve the story draft to fully incorporate the suggestions from the critique.""",
    output_key="current_story",
    mode="single_turn",  # <-- Required for Workflow
)


# Guard node: checks critique & tracks loop iterations to prevent infinite cycles
@node(name="check_critique")
def check_critique(ctx):
    count = ctx.session.state.get("loop_count", 0)
    ctx.session.state["loop_count"] = count + 1
    
    critique = ctx.session.state.get("critique", "")
    
    # Exit loop if approved OR if max_iterations (2) reached
    if "APPROVED" in critique or count >= 2:
        return Event(route="exit")
    return Event(route="refine")

@node(name="finish")
def finish():
    return "Story refinement complete."

root_agent = Workflow(
    name="StoryPipeline",
    edges=[
        # Initial write -> critique -> condition check
        (START, initial_writer_agent, critic_agent, check_critique),
        
        # Route depending on check_critique output:
        (check_critique, {"refine": refiner_agent, "exit": finish}),
        
        # Refiner loops back to critic for review
        (refiner_agent, critic_agent),
    ]
)

print("✅ Loop Workflow created successfully.")


Initial Writer Agent Created!
critic_agent is created!
✅ Loop Workflow created successfully.


In [68]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write an intentionally unfinished, broken draft about an alien: only 25 words, cutting off mid-sentence, with no ending and no moral."
)


InitialWriterAgent > Vessel hull compromised. Atmosphere venting. Subject seven-zero-four approaches the airlock carrying the strange, pulsing artifact. Its compound eyes reflect the dying console’s red-
CriticAgent > Here are 2-3 specific, actionable suggestions for improvement:

1. **Complete the Scene:** The story cuts off abruptly mid-word ("red-"). Ensure the narrative reaches a natural conclusion rather than stopping as a sentence fragment.
2. **Expand the Narrative Arc:** While the snippet sets up a high-stakes cliffhanger, it functions more as a hook than a complete story. Add a beginning and an ending to give the micro-fiction a sense of resolution or a complete thematic arc.
3. **Clarify the Stakes:** Give the reader a brief glimpse into *why* Subject seven-zero-four is carrying the artifact or what the consequences of the hull breach are, adding emotional or narrative weight to the moment.
RefinerAgent > Vessel hull compromised. Atmosphere venting. Subject seven-zero-four ap

### 7. Architectural Decision Framework: Selecting Orchestration Topologies

Choosing the optimal orchestration topology depends on task determinism, causal dependencies, and latency requirements.

![Agent Decision Tree](https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/agent-decision-tree.png)

### Topology Comparison Matrix

| Pattern | Topology Type | When to Use | State Mechanism | Key Advantage |
| :--- | :--- | :--- | :--- | :--- |
| **Supervisor-Subagent** | Dynamic / Stochastic | Open-ended workflows where paths vary by user intent | Dynamic `AgentTool` return values | High conversational flexibility |
| **Sequential** | Deterministic Linear | Ordered multi-stage pipelines ($A \to B \to C$) | `output_key` $\to$ `{template}` injection | Guaranteed step execution |
| **Parallel + Fan-In** | Concurrent Graph | Independent domain tasks requiring low latency | Parallel state keys + `JoinNode` | Multi-core / asynchronous speedup |
| **Evaluator-Optimizer Loop** | Cyclic State Graph | Iterative draft/critique quality refinement | Programmatic routing node + `Event(route=...)` | Self-correcting output quality |

---

### 8. Engineering Summary & Key Takeaways

1. **State as the Integration Bus:**
   Across all topologies, agents do not pass complex memory objects directly to each other. Instead, the runtime's **Session State** acts as a decoupled message bus, populated via `output_key` and referenced via `{placeholder}` syntax.
2. **Deterministic Routing vs. Stochastic Reasoning:**
   Reserve LLM-based routing (`Supervisor`) strictly for situations where user goals are unpredictable. For established business processes, deterministic graph topologies (`Workflow`) drastically reduce latency, token cost, and hallucination rates.
3. **Graph Synchronization (`JoinNode`):**
   Whenever fanning out into parallel sub-agents, always place a synchronization barrier (`JoinNode`) before aggregators to prevent race conditions on session state variables.

#### Core Reference Documentation:
* [Google ADK Agents Overview](https://google.github.io/adk-docs/agents/)
* [Sequential Agents Guide](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
* [Parallel Agents Guide](https://google.github.io/adk-docs/agents/workflow-agents/parallel-agents/)
* [Loop Agents Guide](https://google.github.io/adk-docs/agents/workflow-agents/loop-agents/)
* [ADK Workflow Graphs & Routing](https://google.github.io/adk-docs/agents/workflow-agents/)